# EVE Link Budget Validation: "CNR in 1Hz" Measurement

This notebook measures the carrier-to-noise ratio in a 1 Hz bandwidth (CNR in 1Hz) from the
four CAMRAS Venus radar echo recordings of 22 March 2025, and compares the result to the
ORI link budget prediction. 

## Structure

| Part | What it Does | Source |
|------|--------------|--------|
| **Part A** | Load SigMF data + Doppler correction | Telkamp (2025), cells 4–10 of `eve-cw-detect-example.ipynb`, CC BY-SA 4.0 |
| **Part B** | CNR₁Hz radiometric measurement | ORI white paper, Eqs. 11–15 |

## Data required

From https://data.camras.nl/venus/ (5 ksps decimated files, doppler correction csv, CC BY 4.0):

```
dwingeloo_eve_extract_2025_03_22_12_05_39_1299.500MHz_5ksps_ci16_le.sigmf-data
dwingeloo_eve_extract_2025_03_22_12_05_39_1299.500MHz_5ksps_ci16_le.sigmf-meta
dwingeloo_eve_extract_2025_03_22_12_15_39_1299.500MHz_5ksps_ci16_le.sigmf-data
dwingeloo_eve_extract_2025_03_22_12_15_39_1299.500MHz_5ksps_ci16_le.sigmf-meta
dwingeloo_eve_extract_2025_03_22_12_25_39_1299.500MHz_5ksps_ci16_le.sigmf-data
dwingeloo_eve_extract_2025_03_22_12_25_39_1299.500MHz_5ksps_ci16_le.sigmf-meta
dwingeloo_eve_extract_2025_03_22_12_35_39_1299.500MHz_5ksps_ci16_le.sigmf-data
dwingeloo_eve_extract_2025_03_22_12_35_39_1299.500MHz_5ksps_ci16_le.sigmf-meta
dwingeloo_venus_doppler.csv
```

Set `DATA_PATH` below to the directory containing these files.

In [1]:
# Configuration of file locations and predicted CNR in 1Hz
DATA_PATH = "/Users/w5nyv/EVE/venus/data"

SIGMF_FILES = [
    f"{DATA_PATH}/dwingeloo_eve_extract_2025_03_22_12_05_39_1299.500MHz_5ksps_ci16_le.sigmf-meta",
    f"{DATA_PATH}/dwingeloo_eve_extract_2025_03_22_12_15_39_1299.500MHz_5ksps_ci16_le.sigmf-meta",
    f"{DATA_PATH}/dwingeloo_eve_extract_2025_03_22_12_25_39_1299.500MHz_5ksps_ci16_le.sigmf-meta",
    f"{DATA_PATH}/dwingeloo_eve_extract_2025_03_22_12_35_39_1299.500MHz_5ksps_ci16_le.sigmf-meta",
]
DOPPLER_CSV = f"{DATA_PATH}/dwingeloo_venus_doppler.csv"

# Link budget prediction (from ORI EVE Link Budget notebook, Dwingeloo dataclass)
# Pt=1000 W, D=25 m, η=0.69, f=1299.5 MHz, R=41,973,432 km, T_sys=65 K, ρ=0.152
CNR_1HZ_PREDICTED_DB = 0.56

In [7]:
# Imports
import numpy as np
import pandas as pd
import sigmf
from astropy.time import Time
import scipy.stats
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Part A: Doppler correction 
#
# SOURCE: Telkamp, T. / CAMRAS (2025). eve-cw-detect-example.ipynb, cells 4–10.
#         https://data.camras.nl/venus/   Licensed CC BY-SA 4.0
#
# This function is a direct adaptation of Telkamp (2025).  The only changes from
# the original are (a) wrapping it in a function so it can be called for each of
# the four files, and (b) returning the corrected IQ samples together with the
# file metadata needed for Part B. The methodology is exactly the same. 
#
# The correction applies a quadratic phase ramp per sample:
#
#   phi(t) = f0 * t  +  (fdrift * t²) / 2
#
# where f0 = freq_offset[0]  (initial Doppler at file start, from CSV)
#   and fdrift = doppler_rate  (per-second rate, expanded to sample resolution
#                               via numpy.repeat — Telkamp 2025, cell 10)

def load_and_doppler_correct(sigmf_meta_path, doppler_csv_path):
    """
    Load a Dwingeloo SigMF file and apply the CAMRAS Doppler correction.

    Parameters
    ----------
    sigmf_meta_path : str
        Path to the .sigmf-meta file.
    doppler_csv_path : str
        Path to dwingeloo_venus_doppler.csv (one row per UTC second).

    Returns
    -------
    data : np.ndarray (complex128)
        Doppler-corrected IQ samples.
    fs : int
        Sample rate in samples/second.
    start_time : str
        UTC capture start time (ISO 8601).
    expected_freq_offset : float
        Initial Doppler frequency offset in Hz (= freq_offset[0] from CSV).
    """
    # ── Load IQ data (Telkamp 2025, cell 3) ────────────────────────────────
    handle = sigmf.sigmffile.fromfile(sigmf_meta_path)
    data = handle.read_samples()
    fs = int(handle.get_global_info()[handle.SAMPLE_RATE_KEY])
    start_time = handle.get_capture_info(0)["core:datetime"]

    # ── Load Doppler CSV (Telkamp 2025, cell 4) ─────────────────────────────
    doppler_data = pd.read_csv(doppler_csv_path, sep=",")
    doppler_timestamps = Time(list(doppler_data["rx_time_utc"]))
    freq_offset = np.array(list(doppler_data["freq_offset_hz"]))
    doppler_rate = np.array(list(doppler_data["doppler_rate_hz_s"]))

    # ── Align CSV to file start (Telkamp 2025, cell 6) ──────────────────────
    time_astropy = Time(start_time)
    duration = int(len(data) / fs)
    start_index = np.argmin(np.abs(doppler_timestamps - time_astropy))
    freq_offset = freq_offset[start_index : start_index + duration]
    doppler_rate = doppler_rate[start_index : start_index + duration]

    # ── Quadratic phase correction (Telkamp 2025, cell 10) ──────────────────
    expected_freq_offset = freq_offset[0]               # initial Doppler offset
    dr = np.repeat(doppler_rate, len(data) // len(doppler_rate))  # expand to samples
    t_s = np.arange(len(data)) / fs                     # sample-time vector (seconds)
    fdrift  = -dr                                        # sign convention: positive = upward shift
    f0_Hz   = -expected_freq_offset
    phi_Hz  = (fdrift * t_s**2) / 2 + (f0_Hz * t_s)   # integrated instantaneous frequency
    phi_rad = 2 * np.pi * phi_Hz
    data    = data * np.exp(1j * phi_rad)               # apply correction

    return data, fs, start_time, expected_freq_offset

In [4]:
# Part B: CNR in 1Hz radiometric measurement 
#
# SOURCE: ORI white paper "Validation of the ORI Earth-Venus-Earth Link Budget",
#         Section 3, Equations 11–15.
#
# After Doppler correction (Part A) the Venus echo sits near DC.  This function
# computes CNR in 1Hz using a single full-resolution 280-second FFT:
#
#   1. PSD normalised by N^2 is S[k] = |FFT(x)|^2 / N^2        (Eq. 12)
#   2. Noise floor from median of S in 200–600 Hz band       (Eq. 13)
#   3. Signal power from +/- 1 Hz integration around peak       (Eq. 14)
#   4. CNR in 1Hz = P_sig / (N0_bin · T)                        (Eq. 15)
#
# Step 4 is independent of absolute ADC amplitude because numerator and
# denominator are both derived from the same unnormalised PSD S[k].

def measure_cnr_1hz(data, fs, integration_half_bw_hz=1.0):
    """
    Measure CNR in 1Hz from a Doppler-corrected IQ segment.

    Parameters
    ----------
    data : np.ndarray (complex)
        Doppler-corrected IQ samples at rate fs.
    fs : int
        Sample rate (samples/second).
    integration_half_bw_hz : float
        Half-bandwidth for signal integration in Hz (default 1.0 Hz,
        motivated by Estévez 2025 Doppler spread measurement at 1299.5 MHz).

    Returns
    -------
    cnr_1hz_db : float
        CNR in 1 Hz bandwidth in dB.
    n0_bin_db : float
        Noise power spectral density per FFT bin (dB, relative units).
    """
    N = len(data)
    T = N / fs                              # integration time in seconds

    # ── Step 1: Full-resolution PSD (Eq. 12) ────────────────────────────────
    spectrum = np.fft.fftshift(np.fft.fft(data))
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=1.0/fs))   # Hz
    S = np.abs(spectrum)**2 / N**2          # normalised power spectral density

    # ── Step 2: Noise floor from signal-free band (Eq. 13) ──────────────────
    # Use 200–600 Hz from DC: far from Venus echo, free of transmitter leakage
    noise_mask = (np.abs(freq) >= 200) & (np.abs(freq) <= 600)
    N0_bin = np.median(S[noise_mask])       # robust to narrowband RFI spikes
    n0_bin_db = 10 * np.log10(N0_bin)

    # ── Step 3: Integrate signal +/- BW around DC (Eq. 14) ─────────────────────
    # Integration centred at f=0 after Doppler correction.
    # WHY NOT argmax?? the echo is spread over +/- 1 Hz (~560 bins at 1/280 Hz
    # resolution). Per-bin SNR ≈ −3 dB, so argmax within +/- 15 Hz finds noise
    # peaks (~+12 dB above median) rather than the echo. DC-centred integration
    # is unbiased should capture real signal.
    sig_mask = np.abs(freq) <= integration_half_bw_hz
    n_bins = np.sum(sig_mask)
    P_sig = np.sum(S[sig_mask]) - N0_bin * n_bins   # subtract expected noise

    # ── Step 4: CNR in 1Hz (Eq. 15) ─────────────────────────────────────────────
    # N0 per Hz = N0_bin * T  (converts per-bin noise to per-Hz noise density)
    N0_hz = N0_bin * T
    cnr_linear = P_sig / N0_hz
    cnr_1hz_db = 10 * np.log10(max(cnr_linear, 1e-20))  # guard against log(0)

    return cnr_1hz_db, n0_bin_db

In [5]:
# ── Run both parts for all four Dwingeloo files ───────────────────────────────

results = []

for path in SIGMF_FILES:
    # Part A: Doppler correction (Telkamp 2025)
    data_corr, fs, start_time, f_initial = load_and_doppler_correct(path, DOPPLER_CSV)

    # Part B: CNR in 1Hz measurement (ORI white paper Eq. 15)
    cnr_db, n0_bin_db = measure_cnr_1hz(data_corr, fs, integration_half_bw_hz=1.0)

    # UTC time label from filename
    utc = start_time[:19].replace("T", " ")

    results.append({
        "UTC": utc,
        "f_initial_Hz": f_initial,
        "N0_bin_dB": n0_bin_db,
        "CNR_1Hz_dB": cnr_db,
    })
    print(f"{utc}  f_initial={f_initial:+.1f} Hz  "
          f"N0={n0_bin_db:.2f} dB  CNR in 1Hz={cnr_db:+.3f} dB")

# ── Summary statistics ────────────────────────────────────────────────────────

cnr_values = np.array([r["CNR_1Hz_dB"] for r in results])
N = len(cnr_values)
mean_cnr = np.mean(cnr_values)
std_cnr  = np.std(cnr_values, ddof=1)
t_crit   = scipy.stats.t.ppf(0.975, df=N-1)     # two-tailed 95 % CI
ci_95    = t_crit * std_cnr / np.sqrt(N)
residual = mean_cnr - CNR_1HZ_PREDICTED_DB

print()
print(f"{'─'*60}")
print(f"  Mean CNR₁Hz          : {mean_cnr:+.3f} dB")
print(f"  Std dev (N={N}, ddof=1) : +/-{std_cnr:.3f} dB")
print(f"  95 % CI (t_{N-1}={t_crit:.3f})  : +/-{ci_95:.3f} dB")
print(f"  Predicted (link budget): {CNR_1HZ_PREDICTED_DB:+.3f} dB")
print(f"  Residual (meas − pred) : {residual:+.3f} dB")
print()
if abs(residual) <= ci_95:
    print("   Prediction is WITHIN the 95 % confidence interval.")
    print("   Link budget validated against CAMRAS Venus echo data.")
else:
    print(f"   Prediction falls outside 95 % CI by {abs(residual)-ci_95:.3f} dB.")

2025-03-22 12:05:39  f_initial=+334.8 Hz  N0=-126.43 dB  CNR in 1Hz=+1.455 dB
2025-03-22 12:15:39  f_initial=+199.4 Hz  N0=-125.89 dB  CNR in 1Hz=-0.087 dB
2025-03-22 12:25:39  f_initial=+64.8 Hz  N0=-125.78 dB  CNR in 1Hz=+0.285 dB
2025-03-22 12:35:39  f_initial=-68.8 Hz  N0=-126.20 dB  CNR in 1Hz=+0.927 dB

────────────────────────────────────────────────────────────
  Mean CNR₁Hz          : +0.645 dB
  Std dev (N=4, ddof=1) : +/-0.684 dB
  95 % CI (t_3=3.182)  : +/-1.088 dB
  Predicted (link budget): +0.560 dB
  Residual (meas − pred) : +0.085 dB

   Prediction is WITHIN the 95 % confidence interval.
   Link budget validated against CAMRAS Venus echo data.


In [6]:
# ── Table 2 formatted output ──────────────────────────────────────────────────

rows = []
for i, r in enumerate(results, 1):
    rows.append({
        "TX": f"TX{i}",
        "UTC": r["UTC"][11:19],
        "N0 (dB)": f"{r['N0_bin_dB']:.2f}",
        "CNR in 1Hz (dB)": f"{r['CNR_1Hz_dB']:+.3f}",
    })

rows.append({"TX": "Mean",      "UTC": "—",
             "N0 (dB)": "—",    "CNR in 1Hz (dB)": f"{mean_cnr:+.3f}"})
rows.append({"TX": "Std dev",   "UTC": "—",
             "N0 (dB)": "—",    "CNR in 1Hz (dB)": f"+/-{std_cnr:.3f}"})
rows.append({"TX": "95 % CI",   "UTC": "—",
             "N0 (dB)": "—",    "CNR in 1Hz (dB)": f"+/-{ci_95:.3f}"})
rows.append({"TX": "Predicted", "UTC": "—",
             "N0 (dB)": "—",    "CNR in 1Hz (dB)": f"{CNR_1HZ_PREDICTED_DB:+.3f}"})
rows.append({"TX": "Residual",  "UTC": "—",
             "N0 (dB)": "—",    "CNR in 1Hz (dB)": f"{residual:+.3f}"})

df = pd.DataFrame(rows)
print("Table 2. Measured CNR in 1Hz from each of the four CAMRAS Dwingeloo Venus echo transmissions.")
print()
print(df.to_string(index=False))
print()

Table 2. Measured CNR in 1Hz from each of the four CAMRAS Dwingeloo Venus echo transmissions.

       TX      UTC N0 (dB) CNR in 1Hz (dB)
      TX1 12:05:39 -126.43          +1.455
      TX2 12:15:39 -125.89          -0.087
      TX3 12:25:39 -125.78          +0.285
      TX4 12:35:39 -126.20          +0.927
     Mean        —       —          +0.645
  Std dev        —       —        +/-0.684
  95 % CI        —       —        +/-1.088
Predicted        —       —          +0.560
 Residual        —       —          +0.085



## References

**Telkamp, T. / CAMRAS (2025).** *eve-cw-detect-example.ipynb*: Example detection of Venus echo.  
CAMRAS data archive. https://data.camras.nl/venus/ — Licensed CC BY-SA 4.0.  
Cells 4–10 provide the Doppler correction code used verbatim in `load_and_doppler_correct()` above.

**Evans, J.V. et al. (1965).** Radio echo observations of Venus and Mercury at 23 cm wavelength.  
*Astronomical Journal*, 70, 486–501. [Radar albedo ρ = 0.152]

**Estévez, D. (EA4GPZ) (2025).** Analysis of the CAMRAS Venus radar experiment.  
https://destevez.net/2025/04/analysis-of-the-camras-venus-radar-experiment/  
[Doppler spread measurement: ≈90 % of echo power within ±1 Hz at 1299.5 MHz]